# MinCED CRISPR Array Detection

This notebook demonstrates `run_minced`, which scans nucleotide sequences for the characteristic repeat-spacer architecture of CRISPR arrays. Given raw genomic DNA, MinCED reports each detected array with its repeat consensus, individual spacers, and genomic coordinates. CRISPR detection is typically the first step when characterizing a CRISPR system in a newly sequenced microbe or identifying candidate loci for a Cas9 engineering pipeline.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("minced")
display_overview("minced")
display_docs_section("minced", "Background")

# MinCED

MinCED (Mining CRISPRs in Environmental Datasets) is a tool for detecting [CRISPR](https://en.wikipedia.org/wiki/CRISPR) arrays in nucleotide sequences. It identifies repeat-spacer arrays using a heuristic [k-mer](https://en.wikipedia.org/wiki/K-mer) search algorithm derived from the CRISPR Recognition Tool (CRT), returning structured information about each array's repeats, spacers, and their genomic positions.

## Background

**What does this tool measure/predict?**
MinCED detects [CRISPR](https://en.wikipedia.org/wiki/CRISPR) (Clustered Regularly Interspaced Short Palindromic Repeats) arrays in DNA sequences. CRISPR arrays consist of short, conserved repeat sequences separated by unique [spacer sequences](https://en.wikipedia.org/wiki/CRISPR#Spacer_acquisition) derived from past viral infections.

**Why is this important?**
CRISPR arrays are the [adaptive immune](https://en.wikipedia.org/wiki/CRISPR#Immunity) memory of prokaryotes. Each spacer records a past viral encounter, and the repeats serve as structural elements recognized by [CRISPR-associated (Cas) proteins](https://en.wikipedia.org/wiki/Cas9). Detecting CRISPR arrays is essential for:
- Identifying CRISPR-Cas defense systems in genomes
- Characterizing spacer repertoires for phage-host interaction studies
- Validating that generated DNA sequences contain functional CRISPR loci
- Engineering new CRISPR systems by understanding natural array architecture

**Scientific foundation:**
CRISPR arrays have a characteristic structure: direct repeats of 23-47 nucleotides separated by unique spacers of 26-50 nucleotides. MinCED exploits this regularity by searching for repeated k-mers at consistent intervals, then extending and refining candidate arrays. The algorithm is based on CRT (CRISPR Recognition Tool), which uses a seed-and-extend approach analogous to BLAST but tuned for the specific repeat-spacer pattern of CRISPR loci.

## Available tools

In [2]:
display_available_tools("minced")

- **`run_minced()`** — Detect CRISPR arrays in nucleotide sequences using MinCED

### `run_minced`

`run_minced` detects CRISPR arrays in nucleotide sequences using MinCED (Mining CRISPRs in Environmental Datasets). The tool scans for the repeat-spacer structure characteristic of CRISPR loci and returns each array with its repeat sequences, spacer sequences, and positions. The `min_num_repeats` and `min_repeat_length` thresholds control detection sensitivity.

In [3]:
from proto_tools import (
    MincedInput,
    MincedConfig,
    run_minced,
)

In [4]:
# Display input docs
display_api_reference("minced", "input", "run_minced")

# Synthetic CRISPR array: leader + 5 repeat-spacer units + final repeat + trailer
# Each repeat is 36nt, each spacer is 31nt — mimics a real bacterial CRISPR locus
REPEAT = "GTTTTAGAGCTATGCTGTTTTGAATGGTCCCAAAAC"   # 36nt repeat
SPACER_1 = "AAAAAAAACCCCCCCCTTTTTTTTGGGGGGGG"       # 31nt spacer
SPACER_2 = "CCCCCCCCAAAAAAAGGGGGGGTTTTTTTTAA"       # 31nt spacer
SPACER_3 = "TTTTTTTTGGGGGGGGAAAAAAAACCCCCCCC"       # 31nt spacer
SPACER_4 = "GGGGGGGGTTTTTTTTCCCCCCCCAAAAAAAA"       # 31nt spacer
SPACER_5 = "AACCCCGGTTTTAAAACCCCGGGGTTTTAAAA"       # 31nt spacer

crispr_sequence = (
    "ATCGATCGATCGATCGATCGATCGATCG"  # Leader
    + REPEAT + SPACER_1
    + REPEAT + SPACER_2
    + REPEAT + SPACER_3
    + REPEAT + SPACER_4
    + REPEAT + SPACER_5
    + REPEAT                         # Final repeat (no spacer)
    + "ATCGATCGATCGATCGATCGATCGATCG"  # Trailer
)

inputs = MincedInput(
    sequences=[crispr_sequence],
)

**Input** — `MincedInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[str]` | required | Nucleotide sequence(s) to search for CRISPR arrays |

In [5]:
# Display config docs
display_api_reference("minced", "config", "run_minced")

# Require at least 3 repeats of at least 27nt | see docs above for all fields
config = MincedConfig(
    min_num_repeats=3,
    min_repeat_length=27,
)

**Config** — `MincedConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cpu'` | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int | None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `min_num_repeats` | `int` | `3` | Min repeats per array; raise to 4+ for high-confidence |
| `min_repeat_length` | `int` | `23` | Min repeat length in nt; below 23 risks tandem-repeat hits |
| `max_repeat_length` | `int` | `47` | Max repeat length in nt; covers known CRISPR families, raise for unusual loci |
| `min_spacer_length` | `int` | `26` | Min spacer length in nt; lower (~18) for partial or degraded arrays |
| `max_spacer_length` | `int` | `50` | Max spacer length in nt; raise for noncanonical families with longer spacers |

In [6]:
# Run CRISPR array detection
result = run_minced(inputs, config)

Running run_minced [00:00]

In [7]:
# Display output docs
display_api_reference("minced", "output", "run_minced")

# Report detected arrays, repeat counts, and the first few spacers per array
for seq_result in result.results:
    print(f"{seq_result.sequence_id}: {seq_result.num_arrays} CRISPR array(s) found")
    if seq_result.has_crispr:
        for i, array in enumerate(seq_result.crispr_arrays):
            print(f"  Array {i+1}: {array.num_repeats} repeats")
            print(f"  Spacers: {array.spacers[:3]}..." if len(array.spacers) > 3 else f"  Spacers: {array.spacers}")

**Output** — `MincedOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `list[proto_tools.tools.gene_annotation.minced.minced.MincedSequenceResult]` | `[]` | Per-sequence CRISPR array detection results |

test_genome: 1 CRISPR array(s) found
  Array 1: 6 repeats
  Spacers: ['AAAAAAAACCCCCCCCTTTTTTTTGGGGGGGG', 'CCCCCCCCAAAAAAAGGGGGGGTTTTTTTTAA', 'TTTTTTTTGGGGGGGGAAAAAAAACCCCCCCC']...


## Export Results

CRISPR detection results can be exported to CSV or JSON. CSV produces one row per repeat-spacer unit, making it easy to filter or aggregate in spreadsheet tools. JSON preserves the full nested array structure for programmatic consumption.

In [8]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="minced_results", export_path=output_dir, file_format="json")
print(f"CRISPR detection results exported to {output_dir}")

CRISPR detection results exported to example_output
